# FruitCount — Notebook điều phối trên Kaggle

Notebook này chỉ **điều phối**: toàn bộ logic nằm trong `src/` và `scripts/`.

Trước khi chạy:
1. Upload thư mục `FruitCount/` thành một Kaggle Dataset (ví dụ `fruitcount-code`) và Add Data vào notebook.
2. Add Data bộ MinneApple (hoặc bộ đếm trái cây khác).
3. Settings → Accelerator: GPU (T4/P100), Internet: On (để tải trọng số pretrained).

Kaggle chỉ giữ `/kaggle/working` sau khi chạy xong, nên mọi kết quả được ghi vào đó.

In [ ]:
import os, shutil, sys

# Use a conservative working-directory budget, distinct from GPU VRAM/RAM.
from pathlib import Path
WORKING_ROOT = Path('/kaggle/working')
if not WORKING_ROOT.is_dir():
    raise RuntimeError('This notebook targets Kaggle; /kaggle/working is unavailable.')
os.environ['FRUITCOUNT_WORKING_ROOT'] = str(WORKING_ROOT)
os.environ['FRUITCOUNT_DISK_BUDGET_GB'] = '18'
os.environ['FRUITCOUNT_MIN_FREE_GB'] = '2'
os.environ['HF_HOME'] = str(WORKING_ROOT / '.cache/huggingface')
os.environ['TORCH_HOME'] = str(WORKING_ROOT / '.cache/torch')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PIP_NO_CACHE_DIR'] = '1'
print('Working disk free (GB):', shutil.disk_usage(WORKING_ROOT).free / 10**9)

CODE_INPUT = '/kaggle/input/fruitcount-code/FruitCount'   # sửa theo tên dataset của bạn
WORK = '/kaggle/working/FruitCount'
if not os.path.exists(WORK):
    shutil.copytree(CODE_INPUT, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)

!pip install -q ultralytics pyyaml
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 1. Cấu hình dữ liệu
Sửa đường dẫn cho đúng cấu trúc bộ dữ liệu bạn Add vào (xem `/kaggle/input`).

In [ ]:
from src.utils import load_yaml, save_yaml
from pathlib import Path

ds = load_yaml('configs/dataset.yaml')
# Set both paths explicitly if more than one dataset is attached.
IMAGES_DIR = None
MASKS_DIR = None
if IMAGES_DIR is None or MASKS_DIR is None:
    candidates = [p for p in Path('/kaggle/input').rglob('images')
                  if p.is_dir() and (p.parent / 'masks').is_dir()
                  and 'test' not in p.parts and 'val' not in p.parts]
    train_candidates = [p for p in candidates if 'train' in p.parts]
    candidates = train_candidates or candidates
    if len(candidates) != 1:
        raise ValueError(f'Set IMAGES_DIR and MASKS_DIR explicitly. Found image/mask pairs: {candidates}')
    IMAGES_DIR = str(candidates[0])
    MASKS_DIR = str(candidates[0].parent / 'masks')
for folder in (IMAGES_DIR, MASKS_DIR):
    if not Path(folder).is_dir():
        raise FileNotFoundError(f'Dataset directory does not exist: {folder}')
ds['paths']['images_dir'] = IMAGES_DIR
ds['paths']['masks_dir'] = MASKS_DIR
ds['processed_dir'] = '/kaggle/working/processed'
ds.setdefault('yolo_export', {})['symlink'] = True  # Reuse images from /kaggle/input, no image copies.
save_yaml(ds, 'configs/dataset.yaml')
print(ds['paths'])


In [ ]:
import subprocess
import shlex
import sys

def run_python(arguments):
    subprocess.run([sys.executable, *shlex.split(arguments)], check=True)

run_python('scripts/prepare_data.py')

def run_model(model_key, arguments):
    config = load_yaml(f'configs/models/{model_key}.yaml')
    executable = config.get('python_executable') or sys.executable
    # Stream complete lines into the notebook output, including one line per epoch.
    command = [executable, '-u', *shlex.split(arguments)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, errors='replace') as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    from src.utils.resources import guard_disk
    usage = guard_disk()
    if usage:
        print(f"Working: {usage['used_gb']:.2f} GB used; {usage['remaining_budget_gb']:.2f} GB left in budget")



## 2. Huấn luyện
Kaggle giới hạn ~9–12 giờ/phiên và có thể ngắt; `--resume` tiếp tục từ `last.pt`. Chạy từng model ở các cell/phiên riêng nếu cần.

### Reports across sessions
`TRAIN_MODELS` controls training only. Set `RESUME_INPUT_DIRS` to previous Kaggle Outputs; saved models are discovered automatically. Keep the full runs directory including weights, training and evaluation.
Old test metrics are reused only for the same dataset and protocol. Set `REEVALUATE_OLD_MODELS=True` to evaluate again. Missing metrics trigger evaluation. Missing epoch history cannot be reconstructed from weights.
Set `GENERATE_REPORT_PREDICTIONS=False` to skip loading old models for sample images. Restored checkpoints count toward the disk budget. For overlapping sources, the existing local checkpoint wins.


In [ ]:
TRAIN_MODELS = ['yolov8', 'csrnet', 'yolo11', 'faster_rcnn', 'mask_rcnn']          # thêm: 'yolo11', 'faster_rcnn', 'mask_rcnn'
RESUME_INPUT_DIRS = []  # Previous Kaggle Outputs containing runs/ or model folders.
REPORT_MODELS_EXTRA = []  # Optional extra config keys, e.g. pretrained models.
REEVALUATE_OLD_MODELS = False  # Reuse requires the same dataset and protocol.
GENERATE_REPORT_PREDICTIONS = True  # False skips loading old models for sample images.
EPOCHS = 40                           # ví dụ 5 để chạy thử nhanh

from pathlib import Path

LOW_MEMORY_MODE = True  # Batch 1, no DataLoader workers; keep model architecture unchanged.
SMOKE_TEST_NEW_MODELS = True  # Test real checkpoint before starting the long training loop.

# Checkpoints are saved after each epoch by the training code.
# To resume in a new Kaggle session, restore previous runs here first.
# Optional model keys: rtdetrv2, rf_detr, grounding_dino, countgd, va_count.
# The last two also require source/dependency setup described in Docs/Extra_Models.md.
from src.utils.report_sessions import discover_models, test_report
RUNS = Path('/kaggle/working/runs')
RUNS.mkdir(parents=True, exist_ok=True)
sources = list(dict.fromkeys([*RESUME_INPUT_DIRS, *([RESUME_INPUT_DIR] if globals().get('RESUME_INPUT_DIR', '').strip() else [])]))
for source in sources:
    if TRAIN_MODELS:
        subprocess.run([sys.executable, 'scripts/restore_runs.py', '--source', source,
                        '--out', str(RUNS), '--models', *TRAIN_MODELS], check=True)
    old_keys = [k for k in discover_models('configs/models', [source]) if k not in TRAIN_MODELS]
    if old_keys:
        subprocess.run([sys.executable, 'scripts/restore_runs.py', '--source', source,
                        '--out', str(RUNS), '--models', *old_keys, '--report-only'], check=True)
REPORT_MODELS = list(dict.fromkeys([*TRAIN_MODELS, *discover_models('configs/models', [RUNS]), *REPORT_MODELS_EXTRA]))
EVAL_MODELS = [k for k in REPORT_MODELS if k in TRAIN_MODELS or REEVALUATE_OLD_MODELS
               or test_report(RUNS / load_yaml(f'configs/models/{k}.yaml')['name']) is None]
ACTIVE_MODELS = list(dict.fromkeys([*TRAIN_MODELS, *EVAL_MODELS,
                                   *(REPORT_MODELS if GENERATE_REPORT_PREDICTIONS else [])]))
print('Train:', TRAIN_MODELS)
print('Report:', REPORT_MODELS)
print('Evaluate:', EVAL_MODELS)
print('Reuse test metrics:', [k for k in REPORT_MODELS if k not in EVAL_MODELS])

new_models = {'rtdetrv2', 'rf_detr', 'grounding_dino', 'va_count'}
if new_models.intersection(ACTIVE_MODELS):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-extra-models.txt'], check=True)
for research_model in ('countgd', 'va_count'):
    if research_model in ACTIVE_MODELS:
        run_python(f'scripts/setup_research_models.py --model {research_model}')

RUNS = Path('/kaggle/working/runs')
if 'countgd' in ACTIVE_MODELS:
    countgd_python = load_yaml('configs/models/countgd.yaml').get('python_executable')
    if not countgd_python or not Path(countgd_python).is_file():
        run_python('scripts/setup_countgd_env.py --env /kaggle/working/.venvs/countgd')

model_keys = ' '.join(TRAIN_MODELS)
if TRAIN_MODELS:
    run_python(f'scripts/check_resources.py --models {model_keys} --working-root /kaggle/working --budget-gb 18 --reserve-gb 2 --check-imports')
if SMOKE_TEST_NEW_MODELS:
    for model_key in TRAIN_MODELS:
        if model_key in {'rtdetrv2', 'rf_detr', 'grounding_dino', 'countgd', 'va_count'}:
            run_model(model_key, f'scripts/check_model.py --model configs/models/{model_key}.yaml --out /kaggle/working/runs --backward')
memory_args = '--batch-size 1 --num-workers 0' if LOW_MEMORY_MODE else ''
for m in TRAIN_MODELS:
    cfg = load_yaml(f'configs/models/{m}.yaml')
    if cfg.get('mode') == 'pretrained':
        print(f'{m}: pretrained inference; training skipped')
        continue
    checkpoint = RUNS / cfg['name'] / 'weights' / 'last.pt'
    resume = '--resume' if checkpoint.is_file() else ''
    extra = f'--epochs {EPOCHS}' if EPOCHS else ''
    print(f'{m}: resume {checkpoint}' if resume else f'{m}: start training')
    run_model(m, f'scripts/train.py --model configs/models/{m}.yaml --out {RUNS} {extra} {resume} {memory_args}')


### Loss & validation counting accuracy
Display the saved epoch curves for each selected model immediately after training.


In [ ]:
from IPython.display import Image, display
from pathlib import Path

for m in REPORT_MODELS:
    model_name = load_yaml(f'configs/models/{m}.yaml')['name']
    if load_yaml(f'configs/models/{m}.yaml').get('mode') == 'pretrained':
        print(f'{model_name}: pretrained inference, no training curves')
        continue
    training_dir = Path('/kaggle/working/runs') / model_name / 'training'
    print(f'{model_name}: training loss and validation counting accuracy')
    for filename in ('loss_curves.png', 'accuracy_curves.png'):
        image_path = training_dir / filename
        if image_path.exists():
            display(Image(filename=str(image_path)))
        else:
            print(f'Missing {image_path}; train with the updated code to record epoch history.')


## 3. Đánh giá trên tập test

In [ ]:
for m in EVAL_MODELS:
    run_model(m, f'scripts/evaluate.py --model configs/models/{m}.yaml --out /kaggle/working/runs --split test --batch-size 1 --speed')

# YOLO + tiling (SAHI-style)
if 'yolov8' in EVAL_MODELS:
    run_python(f'scripts/evaluate.py --model configs/models/yolov8.yaml --out /kaggle/working/runs --split test --batch-size 1 --tile --tag _tile --speed')


In [ ]:
run_python(f'scripts/compare.py --runs /kaggle/working/runs --split test')

import pandas as pd
comparison = pd.read_csv('/kaggle/working/runs/comparison/comparison_test.csv')
pd.set_option('display.max_columns', None)
from IPython.display import display
display(comparison.style.format(precision=3, na_rep='\u2014').set_caption(
    'Accuracy / Precision / Recall / F1 / mAP: 0-1; MAPE: %; latency: ms. '
    'Precision and Recall at best F1 (IoU 0.5).'
))
# Separate deployment table: accuracy, speed, storage, and GPU allocation.
deployment = pd.read_csv('/kaggle/working/runs/comparison/deployment_test.csv')
display(deployment.style.format(precision=3, na_rep='\u2014').set_caption(
    'Deployment: latency in ms, sizes in MiB; compare measurements on the same hardware.'))


## 4. Phân tích: quả chồng lấn nhiều thì model nào đếm tốt hơn?

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/runs/comparison/error_by_count_test.png'))
display(Image('/kaggle/working/runs/comparison/comparison_test.png'))

## 5. Ví dụ minh họa (cho báo cáo)

In [ ]:
import glob, random
from src.data import load_split_records

test = load_split_records('/kaggle/working/processed', 'test')
samples = random.Random(0).sample(test, min(4, len(test)))
os.makedirs('/kaggle/working/samples', exist_ok=True)
for r in samples:
    shutil.copy(r.image_path, f'/kaggle/working/samples/{r.image_id}.png')
    print(r.image_id, 'GT =', r.count)

for m in (REPORT_MODELS if GENERATE_REPORT_PREDICTIONS else []):
    run_model(m, f'scripts/predict.py --model configs/models/{m}.yaml --runs /kaggle/working/runs --source /kaggle/working/samples')


In [ ]:
if GENERATE_REPORT_PREDICTIONS:
    import matplotlib.pyplot as plt, matplotlib.image as mpimg

    ids = [r.image_id for r in samples]
    fig, axes = plt.subplots(len(ids), len(REPORT_MODELS), figsize=(6 * len(REPORT_MODELS), 4 * len(ids)), squeeze=False)
    for i, iid in enumerate(ids):
        for j, m in enumerate(REPORT_MODELS):
            model_name = load_yaml(f'configs/models/{m}.yaml')['name']
            axes[i][j].imshow(mpimg.imread(f'/kaggle/working/runs/{model_name}/predictions/{iid}_count.jpg'))
            axes[i][j].set_title(m); axes[i][j].axis('off')
    plt.tight_layout(); plt.show()